# Influence Lines and Surfaces

This notebook demonstrates the recommended influence-analysis workflow in `ospgrillage`:

- multi-lane influence lines in one result object/file,
- 2D and path-overlay influence-line plotting,
- station-based influence-surface extraction,
- physical-coordinate influence-surface plotting for skewed/curved geometry.

In [ ]:
import ospgrillage as og

## Build a compact bridge model

In [ ]:
concrete = og.create_material(material="concrete", code="AS5100-2017", grade="50MPa")
main_section = og.create_section(A=0.896, J=0.133, Iy=0.213, Iz=0.259, Ay=0.233, Az=0.58)
slab_section = og.create_section(A=0.04428, J=2.6e-4, Iy=1.1e-4, Iz=2.42e-4, Ay=3.69e-1, Az=3.69e-1, unit_width=True)
edge_section = og.create_section(A=0.044625, J=2.28e-3, Iy=2.23e-1, Iz=1.2e-3, Ay=3.72e-2, Az=3.72e-2)

main_beam = og.create_member(member_name="Main Beam", section=main_section, material=concrete)
slab = og.create_member(member_name="Slab", section=slab_section, material=concrete)
edge_beam = og.create_member(member_name="Edge Beam", section=edge_section, material=concrete)

bridge = og.create_grillage(
    bridge_name="Influence Demo",
    long_dim=10,
    width=7,
    skew=-20,
    num_long_grid=7,
    num_trans_grid=5,
    edge_beam_dist=1,
    mesh_type="Ortho",
)
bridge.set_member(main_beam, member="interior_main_beam")
bridge.set_member(edge_beam, member="exterior_main_beam_1")
bridge.set_member(edge_beam, member="exterior_main_beam_2")
bridge.set_member(edge_beam, member="edge_beam")
bridge.set_member(slab, member="transverse_slab")
bridge.set_member(edge_beam, member="start_edge")
bridge.set_member(edge_beam, member="end_edge")
bridge.create_osp_model(pyfile=False)

target_element = bridge.get_element(member="interior_main_beam", options="elements")[0]
target_element

## Influence lines from multiple lane paths

In [ ]:
paths = {
    "Lane 1": og.Path(start_point=og.Point(0, 0, 1.5), end_point=og.Point(10, 0, 1.5), increments=11),
    "Lane 2": og.Path(start_point=og.Point(0, 0, 3.5), end_point=og.Point(10, 0, 3.5), increments=11),
    "Lane 3": og.Path(start_point=og.Point(0, 0, 5.5), end_point=og.Point(10, 0, 5.5), increments=11),
}

ils = bridge.analyze_influence_lines(
    paths=paths,
    axle_load=1.0,
    shape_function="hermite",
)
ils

In [ ]:
ils.plot(
    array="forces",
    component="Mz_j",
    element=target_element,
    load_coord="station",
    title="Influence lines by lane (station abscissa)",
    ylabel="Mz_j ordinate",
)

In [ ]:
ils.plot(
    array="forces",
    component="Mz_j",
    element=target_element,
    backend="plotly",
    view="path",
    load_coord="station",
    title="Influence lines overlaid on axle paths",
)

## Influence surface from admissible station grid

In [ ]:
iss = bridge.analyze_influence_surfaces(
    name="Deck IS",
    point_load=1.0,
    shape_function="hermite",
)
iss

In [ ]:
iss.plot(
    array="forces",
    component="Mz_j",
    element=target_element,
    x_coord="longitudinal_station",
    y_coord="transverse_station",
    coordinate_space="station",
    title="Influence surface (station space)",
)

In [ ]:
iss.plot(
    array="forces",
    component="Mz_j",
    element=target_element,
    x_coord="longitudinal_station",
    y_coord="transverse_station",
    coordinate_space="physical",
    backend="plotly",
    view="surface3d",
    title="Influence surface mapped to physical deck coordinates",
)

## Save combined influence outputs

Both result objects retain save metadata and can be written directly to single NetCDF files.

In [ ]:
ils.save("lane_ils.nc")
iss.save("deck_is.nc")